In [2]:
import tensorflow as tf
import numpy as np

# --- Carica il modello tflite float32 ---
interpreter = tf.lite.Interpreter(model_path="model_float.tflite")
interpreter.allocate_tensors()
tensors = interpreter.get_tensor_details()

# --- Helper per estrarre un tensor per shape ---
def get_tensor_by_shape(shape):
    for t in tensors:
        if tuple(t["shape"]) == shape:
            return interpreter.get_tensor(t["index"])
    raise ValueError(f"Tensor con shape {shape} non trovato")

# --- Definizione architettura ---
N_IN = 4    # cambiare in base al tuo input
N_H1 = 16
N_H2 = 8

# --- Individua Normalization constants ---
# Normalization ha due tensori [1,N_IN], uno è mean, uno inv_std
candidates = [get_tensor_by_shape((1, N_IN)), get_tensor_by_shape((1, N_IN))]
# inv_std = tutti > 0, mean = l’altro
if np.all(candidates[0] > 0):
    inv_std = candidates[0].flatten()
    mu = candidates[1].flatten()
else:
    inv_std = candidates[1].flatten()
    mu = candidates[0].flatten()

# --- Estrai pesi e bias dai layer dense ---
W1 = get_tensor_by_shape((N_H1, N_IN))
b1 = get_tensor_by_shape((N_H1,))
W2 = get_tensor_by_shape((N_H2, N_H1))
b2 = get_tensor_by_shape((N_H2,))
W3 = get_tensor_by_shape((1, N_H2)).flatten()
b3 = get_tensor_by_shape((1,)).flatten()[0]

# --- Funzione per generare C array ---
def c_array(name, arr):
    flat = arr.flatten()
    s = f"const float {name}[] = {{\n"
    for i, v in enumerate(flat):
        s += f"  {v:.8e},"
        if (i+1) % 6 == 0:
            s += "\n"
    s += "\n};\n\n"
    return s

# --- Scrive params.h ---
with open("params.h", "w") as f:
    f.write("// AUTO-GENERATED FROM TFLITE\n\n")
    f.write(f"#define N_IN {N_IN}\n")
    f.write(f"#define N_H1 {N_H1}\n")
    f.write(f"#define N_H2 {N_H2}\n\n")
    
    f.write(c_array("mu", mu))
    f.write(c_array("inv_std", inv_std))
    f.write(c_array("W1", W1))
    f.write(c_array("b1", b1))
    f.write(c_array("W2", W2))
    f.write(c_array("b2", b2))
    f.write(c_array("W3", W3))
    f.write(f"const float b3 = {b3:.8e};\n")

print("params.h generato!")


params.h generato!


c:\Users\gbell\OneDrive - Delft University of Technology\Battery\ML models\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
